# 4 - GOLD: 
## Aplicação de Regras de Negócio

>> RAW (CSV original)
        │
        ▼
>> BRONZE (Parquet 1:1 com o CSV)
        │
        ▼
>> SILVER (dados padronizados, limpos e enriquecidos)
        │
        ▼
>  GOLD (indicadores e agregações)

## 4.0 - Bibliotecas

In [1]:
from dotenv import load_dotenv
from datetime import datetime
from src.data.utils import *

load_dotenv()
session = iniciar_cessao_aws()
s3 = session.client("s3")
BUCKET = os.getenv("BUCKET_NAME")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


## 4.1 Concatenando as tabelas, independente do ano

In [2]:
TABELAS = [
    "TS_ALUNO",
    "TS_ESTADO",
    "TS_MUNICIPIO",
    "TS_ITEM",
    "RESULTADOS_METAS",
    "RESULTADOS_METAS_MUNICIPIO"
]

# anos de interesse - poderia pegar diretamente do nome do arquivo, mas para fins de teste, vamos filtrar por ano
ANOS = [2023, 2024, 2025]

In [3]:
for tabela in TABELAS:

    print(f"\n{'='*60}")
    print(f"Consolidando {tabela}")
    print(f"{'='*60}")

    dfs = []

    for ano in ANOS:

        print(f"Lendo {tabela} - {ano}")

        df = carregar_parquet_s3(
            s3_client=s3,
            bucket=BUCKET,
            ano=ano,
            nome_tabela=tabela,
            camada="silver"
        ).head(10)

        # Adiciona metadado do ano
        df["ANO"] = ano

        dfs.append(df)

    # Consolida os anos
    df_gold = pd.concat(
        dfs,
        ignore_index=True,
        sort=False
    )

    print(f"Total de registros: {len(df_gold):,}")

    # Salva na Gold
    salvar_parquet_s3(
        s3_client=s3,
        bucket=BUCKET,
        df=df_gold,
        ano="historico",
        tabela=tabela,
        camada="gold"
    )

    print(f"{tabela} consolidada com sucesso.")


Consolidando TS_ALUNO
Lendo TS_ALUNO - 2023
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2023/dados/TS_ALUNO.parquet
Lendo TS_ALUNO - 2024
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2024/dados/TS_ALUNO.parquet
Lendo TS_ALUNO - 2025
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2025/dados/TS_ALUNO.parquet
Total de registros: 30
Arquivo salvo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ALUNO.parquet
TS_ALUNO consolidada com sucesso.

Consolidando TS_ESTADO
Lendo TS_ESTADO - 2023
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2023/dados/TS_ESTADO.parquet
Lendo TS_ESTADO - 2024
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2024/dados/TS_ESTADO.parquet
Lendo TS_ESTADO - 2025
Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2025/dados/TS_ESTADO.parquet
Total de registros: 30
Arquivo salvo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ESTADO.

In [16]:
df_1 = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_ALUNO",
    camada="gold"
).head(5)



df_2 = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_MUNICIPIO",
    camada="gold"
).head(5)



df_3 = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_ESTADO",
    camada="gold"
).head(5)



df_4 = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="TS_ITEM",
    camada="gold"
).head(5)



df_5 = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="RESULTADOS_METAS",
    camada="gold"
).head(5)



df_6 = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano="historico",
    nome_tabela="RESULTADOS_METAS_MUNICIPIO",
    camada="gold"
).head(5)



Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ALUNO.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_MUNICIPIO.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ESTADO.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/TS_ITEM.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/RESULTADOS_METAS.parquet
Lendo: s3://fiap-postech-challenge-datascience-002/gold/ano=historico/dados/RESULTADOS_METAS_MUNICIPIO.parquet


In [10]:
df_1.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,...,TX_GABARITO_BLOCO_1,CO_BLOCO_2,TX_RESPOSTA_BLOCO_2,TX_GABARITO_BLOCO_2,CO_BLOCO_3,TX_RESPOSTA_BLOCO_3,TX_GABARITO_BLOCO_3,CO_BLOCO_4,TX_RESPOSTA_BLOCO_4,TX_GABARITO_BLOCO_4
0,2023,11,RO,11008701,2,60000001.0,3.0,1100205.0,Porto Velho,0,...,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>
1,2023,11,RO,11008695,2,60000001.0,3.0,1100205.0,Porto Velho,1,...,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>
2,2023,11,RO,11008687,2,60000001.0,3.0,1100205.0,Porto Velho,0,...,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>
3,2023,11,RO,11008682,2,60000001.0,3.0,1100205.0,Porto Velho,1,...,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>
4,2023,11,RO,11008729,2,60000001.0,3.0,1100205.0,Porto Velho,0,...,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>,NaN,<NA>,<NA>


In [11]:
df_2.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,DESC_SERIE,ANO,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,2,5,64.55,758.3304,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,11,RO,1100015,Alta Floresta D'Oeste,2,3,64.55,758.3304,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,11,RO,1100023,Ariquemes,2,3,62.30,757.0999,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,11,RO,1100023,Ariquemes,2,5,62.30,757.0999,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,11,RO,1100031,Cabixi,2,5,69.10,767.8763,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df_3.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,DESC_SERIE,ANO,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2023,11,RO,2,2,58.65,751.4731,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,11,RO,2,3,65.17,760.1971,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,11,RO,2,5,64.60,759.4357,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,13,AM,2,3,49.20,733.6637,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,13,AM,2,5,52.20,736.4687,2º Ano Do Ensino Fundamental,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_4.head() 

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_BLOCO,NU_POSICAO,CO_ITEM,TP_SERIE,TP_DISCIPLINA,NU_DESCRITOR_HABILIDADE,DS_GABARITO,...,NU_PARAM_B,NU_PARAM_C,NU_PARAM_B1,NU_PARAM_B2,NU_PARAM_B3,NU_PARAM_B4,IN_ITEM_COMUM,DESC_DISCIPLINA,DESC_SERIE,ANO
0,2023,11,RO,3,2,81,2,1,H4,B,...,708.75,0.38,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
1,2023,11,RO,7,6,69,2,1,H1,A,...,737.60,0.25,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
2,2023,11,RO,5,8,112,2,1,H9,D,...,728.75,0.15,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
3,2023,11,RO,7,1,98,2,1,H2,A,...,643.49,0.18,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023
4,2023,11,RO,1,5,109,2,1,H9,D,...,757.80,0.18,NaN,NaN,NaN,NaN,0,Língua Portuguesa,2º Ano Do Ensino Fundamental,2023


In [14]:

df_5.head() 

,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,SAEB_2019,SAEB_2021,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025
0,2023,NaN,<NA>,Brasil,PÚBLICA,54.70,35.77,55.89811,59.897807,63.769871,67.471106,70.966377,74.229525,77.243526,> 80,86.00,NaN,NaN,NaN
1,2023,12.0,AC,Acre,PÚBLICA,52.87,20.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,NaN,NaN
2,2023,27.0,AL,Alagoas,PÚBLICA,39.01,30.04,43.88000,49.700000,55.500000,61.100000,66.500000,71.500000,76.000000,> 80,92.36,NaN,NaN,NaN
3,2023,13.0,AM,Amazonas,PÚBLICA,43.83,28.76,52.20000,56.800000,61.300000,65.600000,69.600000,73.400000,76.900000,> 80,76.14,NaN,NaN,NaN
4,2023,16.0,AP,Amapá,PÚBLICA,24.77,18.77,41.56000,47.600000,53.800000,59.900000,65.600000,70.900000,75.800000,> 80,89.68,NaN,NaN,NaN


In [15]:

df_6.head() 

,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,NIVEIS_ALFABETIZACAO_2023,PC_AVALIADOS_LP,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,CO_NIVEL_ALFABETIZACAO,PC_ALUNO_ALFABETIZADO_2025
0,2023,11.0,RO,1100015.0,Alta Floresta D'Oeste,MUNICIPAL,64.55,67.078601,69.512028,71.841093,74.058634,76.159493,78.140434,80.0,3.0,89.37,NaN,NaN,NaN,NaN
1,2023,11.0,RO,1100023.0,Ariquemes,MUNICIPAL,62.30,65.216878,68.023909,70.706161,73.251889,75.652567,77.902777,80.0,3.0,89.79,NaN,NaN,NaN,NaN
2,2023,11.0,RO,1100031.0,Cabixi,MUNICIPAL,69.10,70.845029,72.530686,74.154438,75.714346,77.209042,78.637700,80.0,3.0,90.48,NaN,NaN,NaN,NaN
3,2023,11.0,RO,1100049.0,Cacoal,MUNICIPAL,62.51,65.390716,68.162817,70.811990,73.326986,75.699642,77.924781,80.0,3.0,84.44,NaN,NaN,NaN,NaN
4,2023,11.0,RO,1100056.0,Cerejeiras,MUNICIPAL,58.53,62.090401,65.525172,68.805091,71.906748,74.812904,77.512445,80.0,2.0,92.12,NaN,NaN,NaN,NaN


In [9]:


import io
import boto3

ano = 2023
session = boto3.Session()
s3 = session.client('s3')
BUCKET_NAME = os.getenv("BUCKET_NAME")
ARQUIVO = 'TS_ALUNO'
PASTA = f"bronze/ano={ano}/dados/{ARQUIVO}"

response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PASTA)
arquivos = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]

print(f"Total de arquivos encontrados: {len(arquivos)}")
for f in arquivos[:5]:
    print(f"  - {f}")


NameError: name 'session' is not defined

In [ ]:

#Leitura dos arquivos em um DataFrame
dfs = []
for chave in arquivos:
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=chave)
    df_part = pd.read_parquet(io.BytesIO(obj['Body'].read()))
    dfs.append(df_part)

dataframe = pd.concat(dfs, ignore_index=True)

print(f"\nTotal de linhas: {len(dataframe):,}")
print(f"Colunas: {dataframe.shape[1]}")
dataframe.head()